In [99]:
import pandas as pd
from pyprojroot import here

In [100]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [101]:
c2020 = pd.read_csv(RAW_DATA_DIR/"population/census_2020/conjunto_de_datos_iter_00CSV20.csv", low_memory=False)
c2010 = pd.read_csv(RAW_DATA_DIR/"population/census_2010/iter_00_cpv2010.csv")

In [102]:
c2020['CVEGEO'] = c2020['ENTIDAD'] * 1000 + c2020['MUN']
c2010['CVEGEO'] = c2010['entidad'] * 1000 + c2010['mun']

In [103]:
c2020['8to30_2020'] = c2020[['P_8A14', 'P_15A17', 'P_18A24']].replace(['*', 'N/D'], '0').fillna(0).astype(int).sum(axis=1)
c2010['8to30_2010'] = c2010[['p_8a14', 'p_15a17', 'p_18a24']].replace(['*', 'N/D'], '0').fillna(0).astype(int).sum(axis=1)

In [104]:
c2020['unemployed_2020'] = c2020['PDESOCUP'].replace(['*', 'N/D'], '0').fillna(0).astype(int)
c2010['unemployed_2010'] = c2010['pdesocup'].replace(['*', 'N/D'], '0').fillna(0).astype(int)

In [105]:
c2020_redux = c2020[['CVEGEO', 'POBTOT', '8to30_2020', 'unemployed_2020']].copy()
c2010_redux = c2010[['CVEGEO', 'pobtot', '8to30_2010', 'unemployed_2010']].copy()

In [106]:
c2020_redux.rename(columns={'POBTOT': '2020', 'anio': 'year'}, inplace=True)
c2010_redux.rename(columns={'pobtot': '2010', 'anio': 'year'}, inplace=True)

In [107]:
cdf2020 = c2020_redux.groupby('CVEGEO')[['2020', '8to30_2020', 'unemployed_2020']].sum()
cdf2010 = c2010_redux.groupby('CVEGEO')[['2010', '8to30_2010', 'unemployed_2010']].sum()

In [108]:
c2020_redux.sample(10)

,CVEGEO,2020,8to30_2020,unemployed_2020
69921,12023,27,7,0
105151,17010,2812,869,42
148266,25012,1,0,0
57627,10032,7,0,0
136342,23004,8,0,0
116695,20080,14,2,0
3459,2001,2,0,0
50127,8054,5,0,0
20098,7012,3,0,0
110306,19012,1,0,0


In [109]:
cdf = pd.merge(cdf2020, cdf2010, on='CVEGEO', how='outer')
cdf2025 = cdf.reset_index().set_index('CVEGEO').copy()

In [110]:
cdf

,2020,8to30_2020,unemployed_2020,2010,8to30_2010,unemployed_2010
CVEGEO,,,,,,
0,126411503,36618764,1161819,112769681.0,36494032.0,2034434.0
1000,1432325,442788,13971,1191191.0,406001.0,31462.0
1001,1900527,569874,20346,1595820.0,528568.0,42040.0
1002,103432,34822,852,91492.0,33762.0,2174.0
1003,116736,35950,758,108560.0,38168.0,3748.0
...,...,...,...,...,...,...
32054,38980,12186,660,37027.0,13212.0,584.0
32055,63394,17948,654,58987.0,18432.0,748.0
32056,299399,87364,3406,276537.0,85928.0,4464.0


In [117]:
cdf2025['grw'] = (cdf2025['2020'] - cdf2025['2010']) / 10
cdf2025['grw_8to24'] = (cdf2025['8to30_2020'] - cdf2025['8to30_2010']) / 10
cdf2025['grw_unemp'] = (cdf2025['unemployed_2020'] - cdf2025['unemployed_2010']) / 10

for year in range(2011, 2020):
    cdf2025[str(year)] = cdf2025['2010'] + cdf2025['grw'] * (year - 2010)
    cdf2025[f"8to30_{year}"] = cdf2025['8to30_2010'] + cdf2025['grw_8to24'] * (year - 2010)
    cdf2025[f"unemployed_{year}"] = cdf2025['unemployed_2010'] + cdf2025['grw_unemp'] * (year - 2010)

for year in range(2021, 2026):
    cdf2025[str(year)] = cdf2025['2020'] + cdf2025['grw'] * (year - 2020)
    cdf2025[f"8to30_{year}"] = cdf2025['8to30_2020'] + cdf2025['grw_8to24'] * (year - 2020)
    cdf2025[f"unemployed_{year}"] = cdf2025['unemployed_2020'] + cdf2025['grw_unemp'] * (year - 2020)

cdf2025 = cdf2025.drop(columns=['grw', 'grw_8to24', 'grw_unemp']).reset_index(drop=True)
result = cdf2025.copy()
# id_cols = ['CVEGEO']  
# result = cdf2025.melt(id_vars=id_cols, var_name='metric_year', value_name='value')
# print(result.sample(10))

In [118]:
result.to_csv(INTERIM_DATA_DIR/"population_2015_2025B.csv")

print(f" {len(result)} municipalities processed")
print(result.sample(10))

 2502 municipalities processed
      CVEGEO    2020  8to30_2020  unemployed_2020      2010  8to30_2010  \
312    10013   54033       16940              546   50535.0     17098.0   
1951   26024    1506         394               20    1626.0       478.0   
416    12030   44520       14180              176   40392.0     15744.0   
774    15095  207397       61784             2228  177167.0     57454.0   
2333   30210   62029       18976              256   54918.0     21278.0   
205     7117   16924        5904               34   13967.0      5596.0   
1566   20537   13420        3526              146   13494.0      4232.0   
2426   31090   15054        4448               80   13711.0      4666.0   
958    18002   30951        8280              110   30566.0      9654.0   
809    16004   29903        8146              168   29379.0      9402.0   

      unemployed_2010      2011  8to30_2011  unemployed_2011  ...      2015  \
312            1080.0   55782.0     17082.2           1026.6  ..